In [1]:
import torch
import torch.nn as nn

# 1. ARCHITECTURE CONFIGURATION
# Tuple: (out_channels, kernel_size, stride)
# List: ["B", num_repeats] -> Residual Block
# "S": Scale Prediction (YOLO Head)
# "U": Upsampling
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 1],
    (128, 3, 2),
    ["B", 2],
    (256, 3, 2),
    ["B", 8],
    (512, 3, 2),
    ["B", 8],
    (1024, 3, 2),
    ["B", 4],
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
    (128, 1, 1),
    "U",
    (128, 1, 1),
    (256, 3, 1),
    "S",
]

# 2. BASIC CONVOLUTIONAL BLOCK
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super().__init__()
        self.bn_act = bn_act
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            bias=not bn_act,
            **kwargs
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)

    def forward(self, x):
        if self.bn_act:
            return self.leaky(self.bn(self.conv(x)))
        return self.conv(x)

# 3. RESIDUAL BLOCK (Darknet-53 style)
class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super().__init__()
        self.layers = nn.ModuleList()

        for _ in range(num_repeats):
            self.layers.append(
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels, kernel_size=3, padding=1),
                )
            )

        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)
        return x

# 4. SCALE PREDICTION (YOLO HEAD)
class ScalePrediction(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(
                2 * in_channels,
                (num_classes + 5) * 3,
                bn_act=False,
                kernel_size=1
            )
        )

    def forward(self, x):
        N, C, H, W = x.shape
        x = self.pred(x)
        x = x.reshape(N, 3, self.num_classes + 5, H, W)
        x = x.permute(0, 1, 3, 4, 2)
        return x

class YOLOv3(nn.Module):
    def __init__(self, in_channels=3, num_classes=8):
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []

        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                route = route_connections.pop()
                if route.shape[2:] != x.shape[2:]:
                    route = torch.nn.functional.interpolate(route, size=x.shape[2:], mode='nearest')

                x = torch.cat([x, route], dim=1)

        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels

        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module

                layers.append(
                    CNNBlock(
                        in_channels,
                        out_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        padding=1 if kernel_size == 3 else 0
                    )
                )
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(
                    ResidualBlock(
                        in_channels,
                        num_repeats=num_repeats
                    )
                )

            elif isinstance(module, str):
                if module == "S":
                    layers.append(
                        ResidualBlock(
                            in_channels,
                            use_residual=False,
                            num_repeats=1
                        )
                    )

                    layers.append(
                        CNNBlock(
                            in_channels,
                            in_channels // 2,
                            kernel_size=1
                        )
                    )

                    layers.append(
                        ScalePrediction(
                            in_channels // 2,
                            self.num_classes
                        )
                    )

                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2, mode="nearest"))
                    in_channels = in_channels * 3

        return layers

In [3]:
import os
import torch
import numpy as np
import torchvision.ops as ops
from torch.utils.data import Dataset
from PIL import Image, ImageDraw, ImageFont



# =========================
# CONFIG
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 8
IMAGE_SIZE = 800
CHECKPOINT_FILE = "/home/zkelias/VisDrone/checkpoint.pth.tar"

VAL_IMG_DIR = "/home/zkelias/VisDrone Data/val/images/"
VAL_ANN_DIR = "/home/zkelias/VisDrone Data/val/annotations/"
SAVE_DIR = "/home/zkelias/VisDrone/val_vis"

NUM_IMAGES_TO_SAVE = 20
CONF_THRESH = 0.15
IOU_THRESH = 0.3

CLASS_NAMES = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle"
]

ANCHORS = torch.tensor([
    [(0.0771, 0.0679), (0.05, 0.0293), (0.0281, 0.0407)],
    [(0.0279, 0.0164), (0.0157, 0.0257), (0.0093, 0.0179)],
    [(0.0150, 0.0105), (0.0064, 0.0100), (0.0036, 0.0057)]
], dtype=torch.float32).to(DEVICE)


# =========================
# DATASET
# =========================
class VisDroneDataset(Dataset):
    def __init__(self, img_dir, ann_dir, image_size=800):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.image_size = image_size
        self.images = sorted([f for f in os.listdir(img_dir) if f.endswith(".jpg")])

        self.valid_classes = [1, 2, 3, 4, 5, 6, 9, 10]
        self.class_map = {orig: i for i, orig in enumerate(self.valid_classes)}

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_name = self.images[index]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.ann_dir, img_name.replace(".jpg", ".txt"))

        image = Image.open(img_path).convert("RGB")
        w_orig, h_orig = image.size

        scale = self.image_size / max(w_orig, h_orig)
        nw, nh = int(w_orig * scale), int(h_orig * scale)

        image_resized = image.resize((nw, nh), Image.Resampling.BICUBIC)
        new_image = Image.new("RGB", (self.image_size, self.image_size), (128, 128, 128))

        pad_x = (self.image_size - nw) // 2
        pad_y = (self.image_size - nh) // 2
        new_image.paste(image_resized, (pad_x, pad_y))

        bboxes = []
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        row = [float(x) for x in line.split(",")]
                    except ValueError:
                        continue

                    orig_cls = int(row[5])
                    if orig_cls in self.valid_classes:
                        new_cls = self.class_map[orig_cls]

                        x1 = row[0]
                        y1 = row[1]
                        bw_orig = row[2]
                        bh_orig = row[3]

                        bw = (bw_orig * scale) / self.image_size
                        bh = (bh_orig * scale) / self.image_size
                        x_center = (x1 * scale + pad_x + (bw_orig * scale) / 2) / self.image_size
                        y_center = (y1 * scale + pad_y + (bh_orig * scale) / 2) / self.image_size

                        bboxes.append([new_cls, x_center, y_center, bw, bh])

        image_tensor = torch.from_numpy(np.array(new_image).transpose(2, 0, 1)).float() / 255.0
        bboxes_tensor = torch.tensor(bboxes, dtype=torch.float32) if bboxes else torch.zeros((0, 5), dtype=torch.float32)

        return image_tensor, bboxes_tensor, img_name


# =========================
# LOAD CHECKPOINT
# =========================
def load_checkpoint(checkpoint_file, model):
    print("=> Loading checkpoint")
    checkpoint = torch.load(checkpoint_file, map_location=DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()


# =========================
# PREDICTIONS
# =========================
def get_nms_predictions(outputs, batch_idx, anchors, conf_thresh=0.15, iou_thresh=0.45):
    raw_preds = []

    for scale_idx, out in enumerate(outputs):
        S = out.shape[2]
        obj_probs = torch.sigmoid(out[batch_idx, ..., 4])
        class_probs = torch.softmax(out[batch_idx, ..., 5:], dim=-1)
        boxes = out[batch_idx, ..., 0:4]

        indices = (obj_probs > conf_thresh).nonzero(as_tuple=False)

        for idx in indices:
            a, j, i = idx[0], idx[1], idx[2]

            conf, cls_idx = torch.max(class_probs[a, j, i], dim=-1)
            score = (obj_probs[a, j, i] * conf).item()

            tx, ty, tw, th = boxes[a, j, i]
            tw = torch.clamp(tw, -10, 10)
            th = torch.clamp(th, -10, 10)

            anchor_w, anchor_h = anchors[scale_idx][a]

            x = (torch.sigmoid(tx) + i) / S
            y = (torch.sigmoid(ty) + j) / S
            w = anchor_w * torch.exp(tw)
            h = anchor_h * torch.exp(th)

            raw_preds.append([x.item(), y.item(), w.item(), h.item(), score, cls_idx.item()])

    if not raw_preds:
        return []

    preds_tensor = torch.tensor(raw_preds, dtype=torch.float32, device=outputs[0].device)

    boxes_xyxy = torch.stack([
        preds_tensor[:, 0] - preds_tensor[:, 2] / 2,
        preds_tensor[:, 1] - preds_tensor[:, 3] / 2,
        preds_tensor[:, 0] + preds_tensor[:, 2] / 2,
        preds_tensor[:, 1] + preds_tensor[:, 3] / 2
    ], dim=1)

    scores = preds_tensor[:, 4]
    class_labels = preds_tensor[:, 5]

    keep_idx = ops.batched_nms(boxes_xyxy, scores, class_labels, iou_threshold=iou_thresh)
    preds_tensor = preds_tensor[keep_idx]

    # optional second filter by final score
    preds_tensor = preds_tensor[preds_tensor[:, 4] > conf_thresh]

    return preds_tensor.tolist()


# =========================
# DRAWING
# =========================
def clamp_box(x1, y1, x2, y2, size):
    x1 = max(0, min(size - 1, x1))
    y1 = max(0, min(size - 1, y1))
    x2 = max(0, min(size - 1, x2))
    y2 = max(0, min(size - 1, y2))
    return x1, y1, x2, y2


def draw_boxes(image_pil, gt_boxes, pred_boxes, class_names):
    draw = ImageDraw.Draw(image_pil)

    try:
        font = ImageFont.load_default()
    except:
        font = None

    # draw GT boxes in green
    for gt in gt_boxes:
        cls, x, y, w, h = gt
        x1 = int((x - w / 2) * IMAGE_SIZE)
        y1 = int((y - h / 2) * IMAGE_SIZE)
        x2 = int((x + w / 2) * IMAGE_SIZE)
        y2 = int((y + h / 2) * IMAGE_SIZE)

        x1, y1, x2, y2 = clamp_box(x1, y1, x2, y2, IMAGE_SIZE)

        draw.rectangle([x1, y1, x2, y2], outline="lime", width=2)
        label = f"GT: {class_names[int(cls)]}"
        draw.text((x1, max(0, y1 - 12)), label, fill="lime", font=font)

    # draw predicted boxes in red
    for pred in pred_boxes:
        x, y, w, h, score, cls = pred
        x1 = int((x - w / 2) * IMAGE_SIZE)
        y1 = int((y - h / 2) * IMAGE_SIZE)
        x2 = int((x + w / 2) * IMAGE_SIZE)
        y2 = int((y + h / 2) * IMAGE_SIZE)

        x1, y1, x2, y2 = clamp_box(x1, y1, x2, y2, IMAGE_SIZE)

        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        label = f"P: {class_names[int(cls)]} {score:.2f}"
        draw.text((x1, min(IMAGE_SIZE - 12, y2 + 2)), label, fill="red", font=font)

    return image_pil


# =========================
# MAIN
# =========================
def main():
    os.makedirs(SAVE_DIR, exist_ok=True)

    model = YOLOv3(num_classes=NUM_CLASSES).to(DEVICE)
    load_checkpoint(CHECKPOINT_FILE, model)

    dataset = VisDroneDataset(
        img_dir=VAL_IMG_DIR,
        ann_dir=VAL_ANN_DIR,
        image_size=IMAGE_SIZE
    )

    model.eval()

    with torch.no_grad():
        for idx in range(min(NUM_IMAGES_TO_SAVE, len(dataset))):
            image_tensor, gt_boxes, img_name = dataset[idx]

            input_tensor = image_tensor.unsqueeze(0).to(DEVICE)
            outputs = model(input_tensor)

            pred_boxes = get_nms_predictions(
                outputs,
                batch_idx=0,
                anchors=ANCHORS,
                conf_thresh=CONF_THRESH,
                iou_thresh=IOU_THRESH
            )

            image_np = (image_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            image_pil = Image.fromarray(image_np)

            drawn = draw_boxes(
                image_pil=image_pil,
                gt_boxes=gt_boxes.tolist(),
                pred_boxes=pred_boxes,
                class_names=CLASS_NAMES
            )

            save_path = os.path.join(SAVE_DIR, f"vis_{idx:03d}_{img_name}")
            drawn.save(save_path)
            print(f"Saved: {save_path}")


if __name__ == "__main__":
    main()

=> Loading checkpoint
Saved: /home/zkelias/VisDrone/val_vis/vis_000_0000001_02999_d_0000005.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_001_0000001_03499_d_0000006.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_002_0000001_03999_d_0000007.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_003_0000001_04527_d_0000008.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_004_0000001_05249_d_0000009.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_005_0000001_05499_d_0000010.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_006_0000001_05999_d_0000011.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_007_0000001_07999_d_0000012.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_008_0000001_08414_d_0000013.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_009_0000021_00000_d_0000001.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_010_0000021_00500_d_0000002.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_011_0000021_00800_d_0000003.jpg
Saved: /home/zkelias/VisDrone/val_vis/vis_012_0000022_00000_d_0000004.jpg
Saved: /home/zke